<a href="https://colab.research.google.com/github/joaonct/susep-pc-market-mapping/blob/main/01_diagnostico_e_ranking_susep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Diagnóstico das Bases // Dataset Assessment

In [6]:
import os
import zipfile
import pandas as pd
from google.colab import drive

# 1. Conecta ao Google Drive // Conect to Google Drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Estagio_Actuary/Dados_brutos'
print("Arquivos encontrados na pasta:", os.listdir(BASE_DIR))
print("=" * 70)

# 2. Inspeciona o conteúdo de BaseCompleta.zip sem extrair tudo // Inspects the contents of BaseCompleta.zip without extracting everything
zip_path = os.path.join(BASE_DIR, 'BaseCompleta.zip')
with zipfile.ZipFile(zip_path, 'r') as z:
    print("\n[1] CONTEÚDO DO BaseCompleta.zip:")
    for info in z.infolist():
        tamanho_mb = info.file_size / (1024 * 1024)
        print(f"  - {info.filename} ({tamanho_mb:.2f} MB)")

# 3. Função auxiliar para ler os .xls do SES (que vêm em HTML ou Excel) // Helper function for reading SES .xls files (which come in HTML or Excel format)
def ler_xls_susep(caminho):
    try:
        tabelas = pd.read_html(caminho, decimal=',', thousands='.')
        return tabelas[0]
    except Exception:
        return pd.read_excel(caminho)

# 4. Inspeciona colunas e primeiras linhas de cada .xls // Inspects columns and the first few rows of each .xls file
arquivos_xls = [
    'ADM_MERCADO.xls',
    'Seguros premios.xls',
    'Seguros sinistros.xls',
    'Resseguros premios ganhos.xls'
]

for nome in arquivos_xls:
    caminho = os.path.join(BASE_DIR, nome)
    df = ler_xls_susep(caminho)
    print("\n" + "=" * 70)
    print(f"[2] ARQUIVO: {nome} | Dimensões: {df.shape}")
    print("Colunas:", list(df.columns))
    print(df.head(3).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos encontrados na pasta: ['BaseCompleta.zip', 'ADM_MERCADO.xls', 'Seguros premios.xls', 'Seguros sinistros.xls', 'Resseguros premios ganhos.xls']

[1] CONTEÚDO DO BaseCompleta.zip:
  - Ses_Administradores.csv (40.01 MB)
  - Ses_campos.csv (0.12 MB)
  - Ses_cias.csv (0.04 MB)
  - Ses_dependencias.csv (0.39 MB)
  - Ses_diversos.csv (0.00 MB)
  - Ses_grupos_economicos.csv (12.26 MB)
  - Ses_limite_ret.csv (20.15 MB)
  - Ses_pl_margem.csv (3.52 MB)
  - ses_prev_trad_resgates.csv (0.42 MB)
  - Ses_prov.csv (1.41 MB)
  - SES_prov_segprev.csv (0.21 MB)
  - Ses_ramos.csv (0.01 MB)
  - Ses_rmovram.csv (279.48 MB)
  - Ses_seg_prov_det.csv (3.72 MB)
  - Ses_seguros.csv (131.06 MB)
  - SES_UF2.csv (451.89 MB)
  - Ses_vgbl_fundos.csv (0.22 MB)
  - Ses_vgbl_resgates.csv (0.47 MB)
  - ses_transferenciasexternas.csv (0.62 MB)
  - ses_valoresresmovgrupos.csv (249.38 MB)

Inspecionar Schemas dos CSVs Internos e Cargos // Inspect Schemas for Internal CSVs and Positions

In [7]:
import csv

def inspecionar_csv_zip(zip_path, nome_arquivo, n=5):
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Encontra o nome exato ignorando maiúsculas/minúsculas // Finds the exact name, ignoring upper/lower case
        mapa = {f.lower(): f for f in z.namelist()}
        alvo = mapa.get(nome_arquivo.lower())
        if not alvo:
            print(f"Arquivo {nome_arquivo} não encontrado no zip.")
            return

        with z.open(alvo) as f:
            # Lê amostra para detectar separador e encoding // Reads a sample to detect the separator and encoding
            amostra = f.read(4096).decode('latin1', errors='replace')
            sep = ';' if amostra.count(';') >= amostra.count(',') else ','
            f.seek(0)
            df = pd.read_csv(f, sep=sep, encoding='latin1', nrows=n, low_memory=False)
            print("\n" + "=" * 70)
            print(f"[CSV] {alvo} (sep='{sep}') | Colunas: {list(df.columns)}")
            print(df.head(n).to_string(index=False))

csvs_chave = [
    'Ses_cias.csv',
    'Ses_ramos.csv',
    'ses_gruposramos.csv',
    'Ses_grupos_economicos.csv',
    'Ses_seguros.csv',
    'Ses_Administradores.csv',
    'ses_contatos.csv'
]

for arq in csvs_chave:
    inspecionar_csv_zip(zip_path, arq, n=5)

# Verifica os cargos únicos no ADM_MERCADO.xls // Check the unique job titles in ADM_MERCADO.xls
df_adm = ler_xls_susep(os.path.join(BASE_DIR, 'ADM_MERCADO.xls'))
print("\n" + "=" * 70)
print("[CARGOS ÚNICOS EM ADM_MERCADO.xls]:")
print(df_adm['Cargo'].value_counts(dropna=False).to_string())


[CSV] Ses_cias.csv (sep=';') | Colunas: ['Coenti', 'Noenti', 'Cogrupo', 'Nogrupo']
 Coenti                                       Noenti Cogrupo  Nogrupo
   3654 SICOOB SEGURADORA DE VIDA E PREVIDÊNCIA S.A.              NaN
   3662                           QI SEGURADORA S.A.              NaN
   3671                         USEBENS SEGUROS S.A.              NaN
   3727                        FAIRWAY SEGUROS S.A.               NaN
   3786                          ARCA SEGURADORA S.A              NaN

[CSV] Ses_ramos.csv (sep=';') | Colunas: ['coramo', 'noramo']
 coramo                                     noramo
   1101 1101 - Seguro Agr sem cob do FESR(RUN OFF)
   1601            1601 - Microsseguros de Pessoas
   2201          2201 - Sobrevivência de Assistido
   1102 1102 - Seguro Agr com cob do FESR(RUN OFF)
   1602              1602 - Microsseguros de Danos

[CSV] ses_gruposramos.csv (sep=';') | Colunas: ['GRAID', 'GRANOME', 'GRACODIGO']
 GRAID                GRANOME  GRACODIGO
    

Mapear Ramos P&C, Cargos Regulatórios e Período Mais Recente // Map P&C Lines of Business, Regulatory Roles, and Most Recent Period

In [8]:
def ler_csv_zip(zip_path, nome_arquivo, usecols=None):
    with zipfile.ZipFile(zip_path, 'r') as z:
        mapa = {f.lower(): f for f in z.namelist()}
        alvo = mapa[nome_arquivo.lower()]
        with z.open(alvo) as f:
            return pd.read_csv(f, sep=';', encoding='latin1', usecols=usecols, low_memory=False)

# 1. Todos os Grupos de Ramos e Ramos para definição do filtro P&C // All Line Groups and Lines for defining the P&C filter
df_grupos_ramos = ler_csv_zip(zip_path, 'ses_gruposramos.csv')
df_ramos = ler_csv_zip(zip_path, 'Ses_ramos.csv')

print("=== [1] GRUPOS DE RAMOS ===")
print(df_grupos_ramos.sort_values('GRACODIGO').to_string(index=False))

print("\n=== [2] TODOS OS RAMOS (Ses_ramos.csv) ===")
print(df_ramos.sort_values('coramo').to_string(index=False))

# 2. Verifica o período (damesano) mais recente em Ses_seguros, Contatos e Administradores // Check the most recent period (year-month) in Ses_seguros, Contatos, and Administradores.
df_seg_datas = ler_csv_zip(zip_path, 'Ses_seguros.csv', usecols=['damesano'])
ultimo_mes_seg = df_seg_datas['damesano'].max()
print(f"\n=== [3] ÚLTIMO MÊS DISPONÍVEL EM Ses_seguros.csv: {ultimo_mes_seg} ===")

# 3. Cargos recentes em ses_contatos.csv e Ses_Administradores.csv // Recent positions in ses_contatos.csv and Ses_Administradores.csv
df_contatos = ler_csv_zip(zip_path, 'ses_contatos.csv')
ultimo_mes_cont = df_contatos['damesano'].max()
print(f"\n=== [4] CARGOS EM ses_contatos.csv (Mês mais recente: {ultimo_mes_cont}) ===")
print(df_contatos[df_contatos['damesano'] == ultimo_mes_cont]['dircargo'].value_counts().to_string())

df_admins_zip = ler_csv_zip(zip_path, 'Ses_Administradores.csv')
ultimo_mes_adm = df_admins_zip['damesano'].max()
print(f"\n=== [5] CARGOS EM Ses_Administradores.csv (Mês mais recente: {ultimo_mes_adm}) ===")
print(df_admins_zip[df_admins_zip['damesano'] == ultimo_mes_adm]['dircargo'].value_counts().to_string())

# 4. Verifica Resseguradoras / Corretoras em Ses_cias.csv e Ses_Cessoes_Recebidas.csv // Check Reinsurers / Brokers in Ses_cias.csv and Ses_Cessoes_Recebidas.csv
df_cias = ler_csv_zip(zip_path, 'Ses_cias.csv')
print("\n=== [6] AMOSTRA DE RESSEGURADORAS / BROKERS EM Ses_cias.csv ===")
filtro_ress = df_cias['Noenti'].str.contains('RESSEGUR|REINSUR|CORRET|BROKER|MUNICH|SWISS|HANNOVER|SCOR|IRB|TERRA BRASIS|AUSTRAL', case=False, na=False)
print(df_cias[filtro_ress].to_string(index=False))

df_cessoes = ler_csv_zip(zip_path, 'Ses_Cessoes_Recebidas.csv', nrows=5) if False else ler_csv_zip(zip_path, 'Ses_Cessoes_Recebidas.csv')
print("\n=== [7] COLUNAS DE Ses_Cessoes_Recebidas.csv ===")
print("Colunas:", list(df_cessoes.columns))
print(df_cessoes.tail(5).to_string(index=False))

=== [1] GRUPOS DE RAMOS ===
 GRAID                     GRANOME  GRACODIGO
     1            01 - Patrimonial          1
     2       02 - Riscos Especiais          2
     3      03 - Responsabilidades          3
     4                 04 - Cascos          4
     5              05 - Automóvel          5
     6            06 - Transportes          6
     7     07 - Riscos Financeiros          7
     8                08 - Crédito          8
     9       09 - Pessoas Coletivo          9
    10           10 - Habitacional         10
    11                  11 - RURAL         11
    12                 12 - Outros         12
    13      13- Pessoas Individual         13
    14              14 - Marítimos         14
    15           15 - Aeronáuticos         15
    16          16 - Microsseguros         16
    17               17 - Petróleo         17
    18              18 - Nucleares         18
    19                  19 - Saúde         19
    20 20 - Aceitações do Exterior         20
    21

Gerar Base Consolidada e Planilha Excel para a Fernanda Chaves // Generate a consolidated dataset and an Excel spreadsheet for Fernanda Chaves

In [9]:
import urllib.parse
import numpy as np
import pandas as pd
import zipfile
import os

BASE_DIR = '/content/drive/MyDrive/Estagio_Actuary/Dados_brutos'
OUT_DIR = '/content/drive/MyDrive/Estagio_Actuary'
zip_path = os.path.join(BASE_DIR, 'BaseCompleta.zip')

def ler_csv_zip(zip_path, nome_arquivo, usecols=None, decimal='.'):
    with zipfile.ZipFile(zip_path, 'r') as z:
        mapa = {f.lower(): f for f in z.namelist()}
        alvo = mapa[nome_arquivo.lower()]
        with z.open(alvo) as f:
            return pd.read_csv(f, sep=';', encoding='latin1', usecols=usecols, decimal=decimal, low_memory=False)

def ler_xls_susep(caminho):
    try:
        return pd.read_html(caminho, decimal=',', thousands='.')[0]
    except Exception:
        return pd.read_excel(caminho)

# -------------------------------------------------------------------------
# 1. CADASTRO DE EMPRESAS E GRUPOS ECONÔMICOS (202609) // REGISTRATION OF COMPANIES AND ECONOMIC GROUPS (202609)
# -------------------------------------------------------------------------
df_cias = ler_csv_zip(zip_path, 'Ses_cias.csv')
df_cias['Coenti'] = pd.to_numeric(df_cias['Coenti'], errors='coerce')
df_cias = df_cias.dropna(subset=['Coenti']).copy()
df_cias['Coenti'] = df_cias['Coenti'].astype(int)
df_cias['Noenti'] = df_cias['Noenti'].astype(str).str.strip()

df_grupos = ler_csv_zip(zip_path, 'Ses_grupos_economicos.csv')
df_grupos['damesano'] = pd.to_numeric(df_grupos['damesano'], errors='coerce')
ultimo_mes = int(df_grupos['damesano'].max())
df_grupos_atual = (
    df_grupos[df_grupos['damesano'] == ultimo_mes][['coenti', 'nogrupo']]
    .drop_duplicates(subset=['coenti'])
    .rename(columns={'coenti': 'Coenti', 'nogrupo': 'Grupo_Economico'})
)
df_grupos_atual['Grupo_Economico'] = df_grupos_atual['Grupo_Economico'].astype(str).str.strip()

# -------------------------------------------------------------------------
# 2. RANKING DE SEGURADORAS DE RAMOS ELEMENTARES (P&C) // RANKING OF PROPERTY & CASUALTY (P&C) INSURANCE COMPANIES
# -------------------------------------------------------------------------
df_seg = ler_csv_zip(
    zip_path, 'Ses_seguros.csv',
    usecols=['damesano', 'coenti', 'coramo', 'premio_direto'],
    decimal=','
)
df_seg['damesano'] = pd.to_numeric(df_seg['damesano'], errors='coerce')
df_seg['coenti'] = pd.to_numeric(df_seg['coenti'], errors='coerce')
df_seg['coramo'] = pd.to_numeric(df_seg['coramo'], errors='coerce')
df_seg['premio_direto'] = pd.to_numeric(df_seg['premio_direto'], errors='coerce').fillna(0.0)

# Filtra últimos 12 meses (202510 a 202609) para eficiência de memória // Filters the last 12 months (202510 to 202609) for memory efficiency
df_seg_12m = df_seg[df_seg['damesano'] >= 202510].copy()

# Regra cirúrgica para Ramos Elementares / Danos (P&C) // Surgical rule for Property & Casualty (P&C) lines
r = df_seg_12m['coramo']
cond_pc = (
    ((r >= 100) & (r < 900)) |                        # 01 Patrimonial a 08 Crédito
    ((r >= 1000) & (r < 1200) & (r != 1198)) |        # 10 Habitacional e 11 Rural (sem 1198 Vida Produtor)
    ((r >= 1400) & (r < 1600)) |                      # 14 Marítimos e 15 Aeronáuticos
    (r == 1602) |                                     # 1602 Microsseguros de Danos
    ((r >= 1700) & (r < 1900))                        # 17 Petróleo e 18 Nucleares
)
df_pc = df_seg_12m[cond_pc].copy()

# Prêmio Direto P&C - Acumulado 2026 (Jan a Set/2026) e Últimos 12 Meses // P&C Direct Written Premium - 2026 YTD (Jan to Sep/2026) and Last 12 Months (LTM)
premio_ytd = (
    df_pc[df_pc['damesano'] >= 202601]
    .groupby('coenti')['premio_direto'].sum()
    .rename('Premio_Direto_PC_2026_YTD')
)
premio_12m = (
    df_pc.groupby('coenti')['premio_direto'].sum()
    .rename('Premio_Direto_PC_12M')
)

df_rank_seg = pd.concat([premio_ytd, premio_12m], axis=1).fillna(0.0).reset_index()
df_rank_seg = df_rank_seg.rename(columns={'coenti': 'Coenti'})
# Mantém seguradoras ativas com emissão P&C no período // Keep active insurers with P&C underwriting in the period
df_rank_seg = df_rank_seg[(df_rank_seg['Premio_Direto_PC_2026_YTD'] > 0) | (df_rank_seg['Premio_Direto_PC_12M'] > 0)]
df_rank_seg = df_rank_seg.merge(df_cias[['Coenti', 'Noenti']], on='Coenti', how='left')
df_rank_seg = df_rank_seg.merge(df_grupos_atual, on='Coenti', how='left')
df_rank_seg['Grupo_Economico'] = df_rank_seg['Grupo_Economico'].fillna('INDEPENDENTE / NÃO INFORMADO')

df_rank_seg = df_rank_seg.sort_values('Premio_Direto_PC_2026_YTD', ascending=False).reset_index(drop=True)
df_rank_seg['Ranking_PC_2026'] = df_rank_seg.index + 1
total_pc_ytd = df_rank_seg['Premio_Direto_PC_2026_YTD'].sum()
df_rank_seg['Share_PC_2026_%'] = (df_rank_seg['Premio_Direto_PC_2026_YTD'] / total_pc_ytd * 100).round(2)

# -------------------------------------------------------------------------
# 3. EXTRAÇÃO DE EXECUTIVOS ESTATUTÁRIOS (202609) // STATUTORY EXECUTIVES EXTRACTION (202609)
# -------------------------------------------------------------------------
df_adm = ler_csv_zip(zip_path, 'Ses_Administradores.csv')
df_adm['damesano'] = pd.to_numeric(df_adm['damesano'], errors='coerce')
df_adm['entcodigofip'] = pd.to_numeric(df_adm['entcodigofip'], errors='coerce')
df_adm_atual = df_adm[df_adm['damesano'] == ultimo_mes].dropna(subset=['entcodigofip']).copy()
df_adm_atual['Coenti'] = df_adm_atual['entcodigofip'].astype(int)
df_adm_atual['pesrazaosocial'] = df_adm_atual['pesrazaosocial'].astype(str).str.strip().str.title()
df_adm_atual['dircargo'] = df_adm_atual['dircargo'].astype(str).str.strip()

df_cont = ler_csv_zip(zip_path, 'ses_contatos.csv')
df_cont['damesano'] = pd.to_numeric(df_cont['damesano'], errors='coerce')
df_cont['entcodigofip'] = pd.to_numeric(df_cont['entcodigofip'], errors='coerce')
df_cont_atual = df_cont[df_cont['damesano'] == ultimo_mes].dropna(subset=['entcodigofip']).copy()
df_cont_atual['Coenti'] = df_cont_atual['entcodigofip'].astype(int)
df_cont_atual['pesrazaosocial'] = df_cont_atual['pesrazaosocial'].astype(str).str.strip().str.title()
df_cont_atual['dircargo'] = df_cont_atual['dircargo'].astype(str).str.strip()

def agrupar_nomes(df, mask):
    sub = df[mask].drop_duplicates(subset=['Coenti', 'pesrazaosocial'])
    return sub.groupby('Coenti')['pesrazaosocial'].apply(lambda x: ' | '.join(x))

# CEO / Presidente
s_ceo = agrupar_nomes(
    df_adm_atual,
    df_adm_atual['dircargo'].str.contains(r'Diretoria\s*-\s*(?:DIRETORA\s+)?PRESIDENTE', case=False, regex=True, na=False)
).rename('CEO_Presidente_SUSEP')

# CFO (Diretor Administrativo-Financeiro)
s_cfo = agrupar_nomes(
    df_cont_atual,
    df_cont_atual['dircargo'].str.contains(r'Administrativo.*financeiro', case=False, regex=True, na=False)
).rename('CFO_Dir_Adm_Fin_SUSEP')

# Atuário Principal (Atuário Responsável Técnico) // Chief Actuary (Appointed Actuary)
s_atuario = agrupar_nomes(
    df_cont_atual,
    df_cont_atual['dircargo'].str.contains(r'ATU[ÁA]RIO RESPONS[ÁA]VEL', case=False, regex=True, na=False)
).rename('Atuario_Responsavel_SUSEP')

# CRO / Riscos (Diretor de Controles Internos - Circ. 249/2004) // CRO / Risk (Internal Controls Officer - Circ. 249/2004)
s_cro_ci = agrupar_nomes(
    df_cont_atual,
    df_cont_atual['dircargo'].str.contains(r'Controles Internos', case=False, regex=True, na=False)
).rename('CRO_Dir_Controles_Riscos_SUSEP')

# Demais Diretores Estatutários (apoio para identificar CRO se não estiver em Controles Internos) // Other Statutory Officers (support to identify the CRO if not listed under Internal Controls)
s_outros_dir = agrupar_nomes(
    df_adm_atual,
    df_adm_atual['dircargo'].str.contains(r'^Diretoria\s*-', case=False, regex=True, na=False) &
    ~df_adm_atual['dircargo'].str.contains(r'PRESIDENTE', case=False, regex=True, na=False)
).rename('Outros_Diretores_SUSEP')

df_execs = pd.concat([s_ceo, s_cfo, s_atuario, s_cro_ci, s_outros_dir], axis=1).reset_index()

def gerar_links_busca(nome_empresa):
    nome_limpo = nome_empresa.split('(')[0].strip()
    q_cro = f'site:linkedin.com/in "{nome_limpo}" ("CRO" OR "Chief Risk Officer" OR "Diretor de Riscos" OR "Risk")'
    q_geral = f'site:linkedin.com/in "{nome_limpo}" ("CEO" OR "CFO" OR "Atuário" OR "Actuary" OR "CRO")'
    link_cro = f"https://www.google.com/search?q={urllib.parse.quote_plus(q_cro)}"
    link_geral = f"https://www.google.com/search?q={urllib.parse.quote_plus(q_geral)}"
    return pd.Series([link_cro, link_geral], index=['Busca_CRO_LinkedIn', 'Busca_Executivos_LinkedIn'])

def enriquecer_tabela(df):
    out = df.merge(df_execs, on='Coenti', how='left')
    links = out['Noenti'].apply(gerar_links_busca)
    out = pd.concat([out, links], axis=1)
    for col in ['Contato_CEO', 'Contato_CFO', 'Contato_Atuario', 'Contato_CRO', 'Observacoes']:
        out[col] = ''
    return out

# Enriquecer Seguradoras P&C // Enrich P&C Insurers
df_seg_final = enriquecer_tabela(df_rank_seg)
cols_seg = [
    'Ranking_PC_2026', 'Coenti', 'Noenti', 'Grupo_Economico',
    'Premio_Direto_PC_2026_YTD', 'Premio_Direto_PC_12M', 'Share_PC_2026_%',
    'CEO_Presidente_SUSEP', 'CFO_Dir_Adm_Fin_SUSEP',
    'Atuario_Responsavel_SUSEP', 'CRO_Dir_Controles_Riscos_SUSEP',
    'Outros_Diretores_SUSEP', 'Busca_CRO_LinkedIn', 'Busca_Executivos_LinkedIn',
    'Contato_CEO', 'Contato_CFO', 'Contato_Atuario', 'Contato_CRO', 'Observacoes'
]
df_seg_final = df_seg_final[cols_seg]

# -------------------------------------------------------------------------
# 4. RESSEGURADORAS LOCAIS, ADMITIDAS/EVENTUAIS E BROKERS DE RESSEGUROS // LOCAL, ADMITTED/OCCASIONAL REINSURERS, AND REINSURANCE BROKERS
# -------------------------------------------------------------------------
df_res_xls = ler_xls_susep(os.path.join(BASE_DIR, 'Resseguros premios ganhos.xls'))
df_res_xls['Coenti'] = pd.to_numeric(df_res_xls['Código SUSEP'], errors='coerce').fillna(0).astype(int)
df_res_xls = df_res_xls[df_res_xls['Coenti'] > 0][['Coenti', 'Prêmios de Resseguro']].rename(
    columns={'Prêmios de Resseguro': 'Premios_Resseguro_2026'}
)

# Resseguradoras Locais (30000 a 39999) // Local Reinsurers (30000 to 39999)
df_res_loc = df_cias[(df_cias['Coenti'] >= 30000) & (df_cias['Coenti'] < 40000)][['Coenti', 'Noenti']].copy()
df_res_loc = df_res_loc[~df_res_loc['Noenti'].str.contains('CANCELAD|SUSPENS|LIQUIDA', case=False, na=False)]
df_res_loc = df_res_loc.merge(df_grupos_atual, on='Coenti', how='left')
df_res_loc = df_res_loc.merge(df_res_xls, on='Coenti', how='left')
df_res_loc['Premios_Resseguro_2026'] = df_res_loc['Premios_Resseguro_2026'].fillna(0)
df_res_loc = df_res_loc.sort_values('Premios_Resseguro_2026', ascending=False).reset_index(drop=True)
df_res_loc = enriquecer_tabela(df_res_loc)

# Resseguradoras Admitidas / Eventuais (40000 a 69999) com movimentação recente em Cessões (2025/2026) // Admitted / Occasional Reinsurers (40000 to 69999) with recent Cession activity (2025/2026)
df_cess = ler_csv_zip(zip_path, 'Ses_Cessoes_Recebidas.csv')
df_cess['damesano'] = pd.to_numeric(df_cess['damesano'], errors='coerce')
df_cess['coenti'] = pd.to_numeric(df_cess['coenti'], errors='coerce')
cess_ativas = (
    df_cess[df_cess['damesano'] >= 202501]
    .groupby('coenti')['cessao'].sum()
    .reset_index()
    .rename(columns={'coenti': 'Coenti', 'cessao': 'Volume_Cessao_2025_2026'})
)
df_res_est = df_cias[(df_cias['Coenti'] >= 40000) & (df_cias['Coenti'] < 70000)][['Coenti', 'Noenti']].copy()
df_res_est = df_res_est[~df_res_est['Noenti'].str.contains('CANCELAD|SUSPENS|LIQUIDA', case=False, na=False)]
df_res_est = df_res_est.merge(cess_ativas, on='Coenti', how='inner')
df_res_est = df_res_est.sort_values('Volume_Cessao_2025_2026', ascending=False).reset_index(drop=True)
df_res_est = enriquecer_tabela(df_res_est)

# Brokers de Resseguros (70000 a 79999) // Reinsures Brokers (70000 a 79999)
df_brokers = df_cias[(df_cias['Coenti'] >= 70000) & (df_cias['Coenti'] < 80000)][['Coenti', 'Noenti']].copy()
df_brokers = df_brokers[~df_brokers['Noenti'].str.contains('CANCELAD|SUSPENS|LIQUIDA', case=False, na=False)]
df_brokers = df_brokers.sort_values('Noenti').reset_index(drop=True)
df_brokers = enriquecer_tabela(df_brokers)

# -------------------------------------------------------------------------
# 5. EXPORTAÇÃO PARA EXCEL MULTI-ABAS // MULTI-SHEET EXCEL EXPORT
# -------------------------------------------------------------------------
arquivo_saida = os.path.join(OUT_DIR, 'mapeamento_mercado_PC_susep_202609.xlsx')
with pd.ExcelWriter(arquivo_saida, engine='openpyxl') as writer:
    df_seg_final.to_excel(writer, sheet_name='Seguradoras_PC_Ranking', index=False)
    df_res_loc.to_excel(writer, sheet_name='Resseguradoras_Locais', index=False)
    df_res_est.to_excel(writer, sheet_name='Resseguradoras_Estrangeiras', index=False)
    df_brokers.to_excel(writer, sheet_name='Brokers_Resseguros', index=False)

print("Arquivo gerado com sucesso em:", arquivo_saida)
print(f"- Seguradoras P&C mapeadas: {len(df_seg_final)}")
print(f"- Resseguradoras Locais: {len(df_res_loc)}")
print(f"- Resseguradoras Estrangeiras (ativas 2025/2026): {len(df_res_est)}")
print(f"- Brokers de Resseguros: {len(df_brokers)}")
print("\nTOP 15 SEGURADORAS P&C (2026 YTD):")
cols_preview = ['Ranking_PC_2026', 'Noenti', 'Grupo_Economico', 'Premio_Direto_PC_2026_YTD', 'CEO_Presidente_SUSEP', 'CFO_Dir_Adm_Fin_SUSEP', 'Atuario_Responsavel_SUSEP']
print(df_seg_final[cols_preview].head(15).to_string(index=False))

Arquivo gerado com sucesso em: /content/drive/MyDrive/Estagio_Actuary/mapeamento_mercado_PC_susep_202609.xlsx
- Seguradoras P&C mapeadas: 100
- Resseguradoras Locais: 18
- Resseguradoras Estrangeiras (ativas 2025/2026): 178
- Brokers de Resseguros: 56

TOP 15 SEGURADORAS P&C (2026 YTD):
 Ranking_PC_2026                                   Noenti     Grupo_Economico  Premio_Direto_PC_2026_YTD                  CEO_Presidente_SUSEP                 CFO_Dir_Adm_Fin_SUSEP          Atuario_Responsavel_SUSEP
               1 PORTO SEGURO COMPANHIA DE SEGUROS GERAIS        PORTO SEGURO               1.666090e+10               Patrícia Chacon Jimenez                Rafael Veneziani Kozma             Anelisa Fortes E Faria
               2             TOKIO MARINE SEGURADORA S.A.        TOKIO MARINE               1.062279e+10                Jose Adalberto Ferrara                  Daniel Dibe Da Silva                     Russiel Moscon
               3                     ALLIANZ SEGUROS S.A.       

Auditoria de Cobertura, Deduplicação e Formatação Executiva do Excel // Coverage Audit, Deduplication, and Executive Excel Formatting

In [10]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# 1. Carrega as abas geradas na Célula 4 // Load the tabs generated in Cell 4
arquivo_excel = os.path.join(OUT_DIR, 'mapeamento_mercado_PC_susep_202609.xlsx')
abas = pd.read_excel(arquivo_excel, sheet_name=None)

df_seg = abas['Seguradoras_PC_Ranking']
df_rloc = abas['Resseguradoras_Locais']
df_rest = abas['Resseguradoras_Estrangeiras']
df_brok = abas['Brokers_Resseguros']

# 2. Deduplica Brokers por razão social normalizada (mantendo o código SUSEP mais recente/maior) // Deduplicate brokers by normalized company name (keeping the most recent/highest SUSEP code)
df_brok['Nome_Norm'] = df_brok['Noenti'].str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()
df_brok = (
    df_brok.sort_values('Coenti', ascending=False)
    .drop_duplicates(subset=['Nome_Norm'])
    .drop(columns=['Nome_Norm'])
    .sort_values('Noenti')
    .reset_index(drop=True)
)

# 3. Sinaliza compartilhamento de CEO dentro do mesmo Grupo Econômico (Seguradoras P&C) // Flag CEO sharing within the same Economic Group (P&C Insurers)
grupos_compartilhados = (
    df_seg.dropna(subset=['CEO_Presidente_SUSEP'])
    .groupby(['Grupo_Economico', 'CEO_Presidente_SUSEP'])['Noenti']
    .transform('count')
)
df_seg['Diretoria_Compartilhada_Grupo'] = np.where(
    (df_seg['Grupo_Economico'] != 'INDEPENDENTE / NÃO INFORMADO') & (grupos_compartilhados > 1),
    'Sim (mesmo grupo)',
    'Não'
)

# Reorganiza colunas de Seguradoras para incluir a flag logo após Grupo_Economico // Reorganize insurer columns to include the flag right after Grupo_Economico
cols_seg_ord = [
    'Ranking_PC_2026', 'Coenti', 'Noenti', 'Grupo_Economico', 'Diretoria_Compartilhada_Grupo',
    'Premio_Direto_PC_2026_YTD', 'Premio_Direto_PC_12M', 'Share_PC_2026_%',
    'CEO_Presidente_SUSEP', 'CFO_Dir_Adm_Fin_SUSEP',
    'Atuario_Responsavel_SUSEP', 'CRO_Dir_Controles_Riscos_SUSEP',
    'Outros_Diretores_SUSEP', 'Busca_CRO_LinkedIn', 'Busca_Executivos_LinkedIn',
    'Contato_CEO', 'Contato_CFO', 'Contato_Atuario', 'Contato_CRO', 'Observacoes'
]
df_seg = df_seg[cols_seg_ord]

# 4. Auditoria de preenchimento por aba // Tab-by-tab completion audit
print("=== AUDITORIA DE COBERTURA ESTATUTÁRIA SUSEP (202609) ===")
for nome_aba, df_temp in [
    ('Seguradoras P&C', df_seg),
    ('Resseguradoras Locais', df_rloc),
    ('Resseguradoras Estrangeiras', df_rest),
    ('Brokers de Resseguros', df_brok)
]:
    n = len(df_temp)
    ceos = df_temp['CEO_Presidente_SUSEP'].notna().sum()
    cfos = df_temp['CFO_Dir_Adm_Fin_SUSEP'].notna().sum()
    atus = df_temp['Atuario_Responsavel_SUSEP'].notna().sum()
    cros = df_temp['CRO_Dir_Controles_Riscos_SUSEP'].notna().sum()
    print(f"\n[{nome_aba}] Total: {n} empresas")
    print(f"  - CEO/Presidente: {ceos}/{n} ({ceos/n*100:.1f}%)")
    print(f"  - CFO (Adm-Fin):  {cfos}/{n} ({cfos/n*100:.1f}%)")
    print(f"  - Atuário RT:     {atus}/{n} ({atus/n*100:.1f}%)")
    print(f"  - CRO/Controles:  {cros}/{n} ({cros/n*100:.1f}%)")

# 5. Salva versão final formatada // Save final formatted version
arquivo_formatado = os.path.join(OUT_DIR, 'mapeamento_mercado_PC_susep_202609_FINAL.xlsx')
with pd.ExcelWriter(arquivo_formatado, engine='openpyxl') as writer:
    df_seg.to_excel(writer, sheet_name='Seguradoras_PC_Ranking', index=False)
    df_rloc.to_excel(writer, sheet_name='Resseguradoras_Locais', index=False)
    df_rest.to_excel(writer, sheet_name='Resseguradoras_Estrangeiras', index=False)
    df_brok.to_excel(writer, sheet_name='Brokers_Resseguros', index=False)

wb = openpyxl.load_workbook(arquivo_formatado)
header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)
link_font = Font(color='0563C1', underline='single')

for ws in wb.worksheets:
    ws.freeze_panes = 'D2'
    ws.auto_filter.ref = ws.dimensions

    headers = {cell.value: idx + 1 for idx, cell in enumerate(ws[1])}
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    # Formata moedas, percentuais e links clicáveis // Format currency, percentages, and clickable links
    for row in range(2, ws.max_row + 1):
        for col_name, col_idx in headers.items():
            c = ws.cell(row=row, column=col_idx)
            if col_name in ['Premio_Direto_PC_2026_YTD', 'Premio_Direto_PC_12M', 'Premios_Resseguro_2026', 'Volume_Cessao_2025_2026']:
                c.number_format = 'R$ #,##0.00'
            elif col_name == 'Share_PC_2026_%':
                c.number_format = '0.00"%"'
            elif col_name in ['Busca_CRO_LinkedIn', 'Busca_Executivos_LinkedIn'] and isinstance(c.value, str) and c.value.startswith('http'):
                url = c.value
                c.hyperlink = url
                c.value = 'Abrir Busca LinkedIn'
                c.font = link_font
                c.alignment = Alignment(horizontal='center')

    # Ajusta largura das colunas com limite máximo para não estourar a tela // Adjust column widths with a maximum limit to avoid overflowing the screen
    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        max_len = max(len(str(ws.cell(row=r, column=col_idx).value or '')) for r in range(1, min(ws.max_row + 1, 50)))
        ws.column_dimensions[col_letter].width = min(max(max_len + 4, 14), 42)

wb.save(arquivo_formatado)
print("\nPlanilha final formatada salva em:", arquivo_formatado)

=== AUDITORIA DE COBERTURA ESTATUTÁRIA SUSEP (202609) ===

[Seguradoras P&C] Total: 100 empresas
  - CEO/Presidente: 66/100 (66.0%)
  - CFO (Adm-Fin):  97/100 (97.0%)
  - Atuário RT:     87/100 (87.0%)
  - CRO/Controles:  94/100 (94.0%)

[Resseguradoras Locais] Total: 18 empresas
  - CEO/Presidente: 10/18 (55.6%)
  - CFO (Adm-Fin):  14/18 (77.8%)
  - Atuário RT:     9/18 (50.0%)
  - CRO/Controles:  13/18 (72.2%)

[Resseguradoras Estrangeiras] Total: 178 empresas
  - CEO/Presidente: 4/178 (2.2%)
  - CFO (Adm-Fin):  6/178 (3.4%)
  - Atuário RT:     0/178 (0.0%)
  - CRO/Controles:  2/178 (1.1%)

[Brokers de Resseguros] Total: 55 empresas
  - CEO/Presidente: 13/55 (23.6%)
  - CFO (Adm-Fin):  13/55 (23.6%)
  - Atuário RT:     6/55 (10.9%)
  - CRO/Controles:  20/55 (36.4%)

Planilha final formatada salva em: /content/drive/MyDrive/Estagio_Actuary/mapeamento_mercado_PC_susep_202609_FINAL.xlsx
